# 2) M2.2 Price formation — Exercises

**Goals:** capture prices and rates, a general fuel-switching formula, which technology sets the price how often, the solar capacity at which cannibalisation bites, and how flexible demand absorbs negative prices — checked with a full-year PyPSA model.

The setup cell rebuilds the lecture's synthetic year, fleet and vectorised clearing. Prices in €/MWh, capacities in MW, energy in MWh.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import pypsa

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- the lecture's synthetic year (same seed, same numbers) ----
rng = np.random.default_rng(7)
idx = pd.date_range("2025-01-01", periods=8760, freq="h")
hour, doy = idx.hour.to_numpy(), idx.dayofyear.to_numpy()
load = (2050 + 350 * np.cos(2 * np.pi * (doy - 15) / 365) + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0)
        - 150 * (idx.dayofweek.to_numpy() >= 5) + rng.normal(0, 60, 8760))
w = np.zeros(8760)
shocks = rng.normal(0, 1, 8760)
for t in range(1, 8760):
    w[t] = 0.97 * w[t - 1] + 0.24 * shocks[t]
wind_cf = np.clip(0.27 + 0.18 * w + 0.06 * np.cos(2 * np.pi * (doy - 15) / 365), 0, 0.95)
sun = np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * (0.65 + 0.35 * np.sin(2 * np.pi * (doy - 80) / 365))
cloud = np.clip(1 - 0.5 * rng.beta(2, 3, 365).repeat(24), 0.2, 1)
solar_cf = np.clip(sun * cloud * 0.95, 0, 1)
ror_cf = np.clip(0.55 + 0.25 * np.sin(2 * np.pi * (doy - 60) / 365), 0.2, 0.9)

COAL = {"fuel": 12.0, "eff": 0.38, "ef": 0.34}
GAS = {"fuel": 35.0, "eff": 0.55, "ef": 0.20}
OCGT = {"fuel": 35.0, "eff": 0.38, "ef": 0.20}


def thermal_mc(plant, co2):
    return (plant["fuel"] + co2 * plant["ef"]) / plant["eff"]


def fleet(wind_MW=1000, solar_MW=1000, co2=70.0, res_bid=-20.0):
    return [("solar", solar_MW * solar_cf, res_bid), ("wind", wind_MW * wind_cf, res_bid),
            ("chp_must_run", np.full(8760, 150.0), -30.0), ("run_of_river", 500 * ror_cf, 0.0),
            ("nuclear_share", np.full(8760, 350.0), 12.0), ("coal", np.full(8760, 600.0), thermal_mc(COAL, co2)),
            ("ccgt", np.full(8760, 900.0), thermal_mc(GAS, co2)), ("imports", np.full(8760, 800.0), thermal_mc(GAS, co2) + 15),
            ("ocgt", np.full(8760, 400.0), thermal_mc(OCGT, co2))]


def clear_year(offers, demand):
    names = [o[0] for o in offers]
    cap = np.vstack([o[1] for o in offers])
    price = np.array([o[2] for o in offers])
    order = np.argsort(price, kind="stable")
    cap, price = cap[order], price[order]
    cum = np.cumsum(cap, axis=0)
    marginal = (cum < demand).sum(axis=0)
    dispatch = np.clip(np.minimum(cap, demand - (cum - cap)), 0, None)
    return price[marginal], pd.DataFrame(dispatch.T, index=idx, columns=[names[i] for i in order])

### Warm-ups

**1. Capture price and capture rate**

Write `capture_rate(price, generation)` returning (capture price, capture rate). Check it on simple cases first.

In [ ]:
def capture_rate(price, generation) -> tuple[float, float]:
    """Generation-weighted average price (EUR/MWh) and its ratio to the time-average price."""
    ...  # YOUR CODE HERE

In [ ]:
assert np.allclose(capture_rate([10, 30], [5, 5]), (20, 1.0)), "flat output earns the average price"
assert np.allclose(capture_rate([10, 30], [1, 0]), (10, 0.5)), "producing only in the cheap hour"
price, dispatch = clear_year(fleet(), load)
cp, cr = capture_rate(price, dispatch.solar)
assert np.isclose(cr, 0.948, atol=1e-3), "the lecture's solar capture rate at 1,000 MW"

**2. A general fuel-switching price**

Write `switching_price(a, b)` for two thermal plants given as dicts (`fuel` €/MWh_th, `eff`, `ef` tCO₂/MWh_th): the CO₂ price at which their marginal costs are equal. Return `None` if they never switch (equal emission intensity per MWh of electricity).

In [ ]:
def switching_price(a: dict, b: dict):
    """CO2 price (EUR/t) at which plants a and b have equal marginal cost, or None."""
    ...  # YOUR CODE HERE

In [ ]:
s = switching_price(COAL, GAS)
assert np.isclose(s, 60.36, atol=0.01) and np.isclose(switching_price(GAS, COAL), s), "symmetric"
assert np.isclose(thermal_mc(COAL, s), thermal_mc(GAS, s))
assert switching_price(GAS, GAS) is None
efficient_coal = {**COAL, "eff": 0.45}
assert switching_price(efficient_coal, GAS) > s, "a more efficient coal plant needs a higher CO2 price to be displaced"
print(f"coal vs gas: {s:.2f} EUR/t | efficient coal vs gas: {switching_price(efficient_coal, GAS):.2f} EUR/t")

### Core

**3. Who sets the price?**

Write `price_setters(offers, demand)` returning the share of hours in which each technology is the **marginal** one (sets the price). Since wind and solar offer at the same price, report them together as `"wind/solar"`. Compare the base system with 2,500 MW of solar.

In [ ]:
def price_setters(offers, demand) -> pd.Series:
    """Share of hours in which each technology (or group with the same offer price) sets the price."""
    ...  # YOUR CODE HERE

In [ ]:
base = price_setters(fleet(), load)
sunny = price_setters(fleet(solar_MW=2500), load)
assert np.isclose(base.sum(), 1) and np.isclose(sunny.sum(), 1)
assert base.index[0] == "ccgt", "gas sets the price most often in the base system"
assert sunny.get("wind/solar", 0) > base.get("wind/solar", 0), "more solar: renewables set the price more often"
pd.DataFrame({"base": base, "solar_2500MW": sunny}).fillna(0).round(3)

**4. When does cannibalisation bite?**

Write `capacity_at_capture_rate(target, tech, step=100)` returning the smallest capacity (MW, on a grid of `step`, starting at `step`) at which the technology's capture rate falls **below** `target`, with the other technology at 1,000 MW. Find it for solar and wind at 0.7.

In [ ]:
def capacity_at_capture_rate(target: float, tech: str, step: int = 100, max_MW: int = 6000):
    """Smallest capacity on the grid where the capture rate drops below target (None if never)."""
    ...  # YOUR CODE HERE

In [ ]:
solar_mw = capacity_at_capture_rate(0.7, "solar")
wind_mw = capacity_at_capture_rate(0.7, "wind")
assert 1000 < solar_mw < 2000 and wind_mw > solar_mw
print(f"capture rate falls below 0.7 at {solar_mw} MW of solar, but only at {wind_mw} MW of wind")

**5. Interpretation**

Your price-setter table shows how often each technology is marginal. Why is the price-setter share a better guide to a technology's *revenue* than its share of energy? Use the 2,500 MW solar case to explain why solar's own growth reduces its revenue per MWh.

*Your answer here.*

### Challenge

**6. Flexible demand absorbs negative prices**

With 2,500 MW of solar, add a 300 MW electrolyser that consumes whenever power costs **at most 30 €/MWh** (a demand bid: in PyPSA a `sign=-1` generator with `marginal_cost=-30`, M2.1). Write `with_electrolyser(electrolyser_MW)` solving the full year in PyPSA and returning a dict with the number of negative-price hours, solar's capture rate, curtailed solar (MWh) and — if present — the electrolyser's full-load hours and average purchase price. Compare 0 and 300 MW.

In [ ]:
def with_electrolyser(electrolyser_MW: float, bid: float = 30.0) -> dict:
    """Full-year PyPSA market with 2,500 MW of solar and an optional flexible electrolyser."""
    ...  # YOUR CODE HERE

In [ ]:
without = with_electrolyser(0)
own_price, _ = clear_year(fleet(solar_MW=2500), load)
assert without["negative_hours"] == int((own_price < 0).sum()), "PyPSA without the electrolyser = the lecture's clearing"
flex = with_electrolyser(300)
assert flex["negative_hours"] < without["negative_hours"] / 2
assert flex["solar_capture_rate"] > without["solar_capture_rate"]
assert flex["solar_curtailed_MWh"] < without["solar_curtailed_MWh"]
assert flex["avg_purchase_price"] < 30 and flex["full_load_hours"] > 500
pd.DataFrame({"no electrolyser": without, "300 MW electrolyser": flex}).round(3)

**7. Interpretation**

Who benefits from the electrolyser — the solar owners, the electrolyser operator, or other consumers? Its average purchase price is far below its 30 €/MWh bid: why? And what does its number of full-load hours mean for whether it would be built without other revenue?

*Your answer here.*